# Prepare Dataset

In [17]:
import json, string
import numpy as np

In [18]:
with open("data/alfred/train.json", "r") as file:
    train_dataset = json.load(file)

with open("data/alfred/valid_unseen.json", "r") as file:
    test_dataset_unseen = json.load(file)

with open("data/alfred/valid_seen.json", "r") as file:
    test_dataset_seen = json.load(file)

with open("data/stopwords.json", "r") as file:
    stopwords = set(json.load(file))

def text_to_words(text):
    return text.strip(string.punctuation + string.whitespace).lower().split()

# Method 1: Bag of Words (BoW)

For each label, consider all the task descriptions as its document content.
In BoW model, we simply count the occurences of each word, without considering the order.

In [19]:
from collections import defaultdict, Counter

# Construct the BoW for each label
BoW = defaultdict(Counter)
for item in train_dataset:
    BoW[item["label"]].update(text_to_words(item["task"]))

# Print the labels
for label in BoW.keys():
    print(label)

look_at_obj_in_light
pick_and_place_simple
pick_and_place_with_movable_recep
pick_clean_then_place_in_recep
pick_cool_then_place_in_recep
pick_heat_then_place_in_recep
pick_two_obj_and_place


## 1.1 BoW Probability

First we consider predicting by the probability of words in each labels. That is:

$$
P(\text{text}|\text{label}) 
= \prod_{\text{word}\in\text{text}} p(\text{label}|\text{word})
= \prod_{\text{word}\in\text{text}} \frac{\text{tf}_{\text{word},\text{label}}}{\text{tf}_{\text{word}}}
$$

Obviously for calculation simplicity we can simply compare:

$$
\begin{aligned}
&\log P(\text{text}|\text{label}) \\
=& \sum_{\text{word}\in\text{text}} \left( \log{\text{tf}_{\text{word},\text{label}}} -\log{\text{tf}_{\text{word}}} \right) \\
=& \sum_{\text{word}\in\text{text}} \log{\text{tf}_{\text{word},\text{label}}}\ 
\underbrace{-\sum_{\text{word}\in\text{text}} \log{\text{tf}_{\text{word}}}}_\text{common part}
\end{aligned}
$$

In [20]:
def bow_prob(text, remove_stopwords=False):
    words = text_to_words(text)
    label_log_probs = {}
    for label, counter in BoW.items():
        log_prob = 0.0
        for word in words:
            if remove_stopwords and word in stopwords:
                continue
            if counter[word] > 0:
                log_prob += np.log(counter[word])
        label_log_probs[label] = log_prob
    return max(label_log_probs, key=label_log_probs.get)

In [21]:
correct = 0
for item in test_dataset_unseen:
    label = item["label"]
    predict = bow_prob(item["task"], remove_stopwords=False)
    if label == predict:
        correct += 1
print(f"Unseen Dataset Accuracy: {correct / len(test_dataset_unseen)}")

Unseen Dataset Accuracy: 0.9013398294762485


In [22]:
correct = 0
for item in test_dataset_seen:
    label = item["label"]
    predict = bow_prob(item["task"], remove_stopwords=False)
    if label == predict:
        correct += 1
print(f"seen Dataset Accuracy: {correct / len(test_dataset_seen)}")

seen Dataset Accuracy: 0.8463414634146341


## Methods 2

**TF-IDF with Supervised Classifiers**

To represent the instruction text numerically, we adopt the Term Frequency–Inverse Document Frequency (TF-IDF) representation. 
TF-IDF reflects the importance of a word in a document relative to the entire corpus.

For a term $t$ in document $d$, the TF-IDF weight is defined as

\begin{equation}
\text{TF-IDF}(t,d) = TF(t,d) \times IDF(t)
\end{equation}

where the term frequency (TF) is defined as

\begin{equation}
TF(t,d) = \frac{f_{t,d}}{\sum_{t'} f_{t',d}}
\end{equation}

Here, $f_{t,d}$ denotes the number of occurrences of term $t$ in document $d$.

The inverse document frequency (IDF) is defined as

\begin{equation}
IDF(t) = \log \left(\frac{N}{df(t)}\right)
\end{equation}

where $N$ is the total number of documents in the corpus and $df(t)$ is the number of documents containing term $t$.

Each document is therefore represented as a TF-IDF feature vector

\begin{equation}
\mathbf{x}_d = [w_1, w_2, \dots, w_V]
\end{equation}

where $V$ denotes the vocabulary size and $w_i$ represents the TF-IDF weight of term $i$.

The resulting TF-IDF feature vectors are used as input to several supervised classifiers, including Logistic Regression, Random Forest, Support Vector Machines (SVM), and k-Nearest Neighbors (KNN), in order to predict the task label from the instruction text.

In [23]:
# Prepare train data
train_texts = [item["task"] for item in train_dataset]
train_labels = [item["label"] for item in train_dataset]

# validation (seen)
test_texts_seen = [item["task"] for item in test_dataset_seen]
test_labels_seen = [item["label"] for item in test_dataset_seen]

# validation (unseen)
test_texts_unseen = [item["task"] for item in test_dataset_unseen]
test_labels_unseen = [item["label"] for item in test_dataset_unseen]

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words=list(stopwords),
    max_features=5000,
    ngram_range=(1,2)
)

X_train = vectorizer.fit_transform(train_texts)

X_test_seen = vectorizer.transform(test_texts_seen)
X_test_unseen = vectorizer.transform(test_texts_unseen)

print("Train shape:", X_train.shape)
print("Seen shape:", X_test_seen.shape)
print("Unseen shape:", X_test_unseen.shape)

Train shape: (21025, 5000)
Seen shape: (820, 5000)
Unseen shape: (821, 5000)


In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score

log_model = LogisticRegression(max_iter=2000)

log_model.fit(X_train, train_labels)

pred_seen = log_model.predict(X_test_seen)
pred_unseen = log_model.predict(X_test_unseen)

acc_seen = accuracy_score(test_labels_seen, pred_seen)
acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("Logistic Regression")
print("Seen Accuracy:", acc_seen)
print("Unseen Accuracy:", acc_unseen)
print()

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, train_labels)

pred_seen = rf_model.predict(X_test_seen)
pred_unseen = rf_model.predict(X_test_unseen)

acc_seen = accuracy_score(test_labels_seen, pred_seen)
acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("Random Forest")
print("Seen Accuracy:", acc_seen)
print("Unseen Accuracy:", acc_unseen)
print()

svm_model = LinearSVC()

svm_model.fit(X_train, train_labels)

pred_seen = svm_model.predict(X_test_seen)
pred_unseen = svm_model.predict(X_test_unseen)

acc_seen = accuracy_score(test_labels_seen, pred_seen)
acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("SVM")
print("Seen Accuracy:", acc_seen)
print("Unseen Accuracy:", acc_unseen)
print()

knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train, train_labels)

pred_seen = knn_model.predict(X_test_seen)
pred_unseen = knn_model.predict(X_test_unseen)

acc_seen = accuracy_score(test_labels_seen, pred_seen)
acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("KNN")
print("Seen Accuracy:", acc_seen)
print("Unseen Accuracy:", acc_unseen)
print()

Logistic Regression
Seen Accuracy: 0.9195121951219513
Unseen Accuracy: 0.9464068209500609

Random Forest
Seen Accuracy: 0.9195121951219513
Unseen Accuracy: 0.9403166869671132

SVM
Seen Accuracy: 0.9280487804878049
Unseen Accuracy: 0.9451887941534713

KNN
Seen Accuracy: 0.8670731707317073
Unseen Accuracy: 0.8708891595615104



In [ ]:
results = {}

def evaluate(model, name):
    model.fit(X_train, train_labels)

    pred_seen = model.predict(X_test_seen)
    pred_unseen = model.predict(X_test_unseen)

    acc_seen = accuracy_score(test_labels_seen, pred_seen)
    acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

    results[name] = (acc_seen, acc_unseen)

evaluate(LogisticRegression(max_iter=2000), "Logistic")
evaluate(RandomForestClassifier(n_estimators=200), "RandomForest")
evaluate(LinearSVC(), "SVM")
evaluate(KNeighborsClassifier(n_neighbors=5), "KNN")

print("Model Comparison")
for k,v in results.items():
    print(k, "Seen:", round(v[0],4), "Unseen:", round(v[1],4))

Model Comparison
Logistic Seen: 0.9195 Unseen: 0.9464
RandomForest Seen: 0.9207 Unseen: 0.9367
SVM Seen: 0.928 Unseen: 0.9452
KNN Seen: 0.8671 Unseen: 0.8709
